# What image does Cellpose actually see? (suite2p 1.1.0)

**The change you're seeing.** Old suite2p chose the Cellpose detection image with an
integer `anatomical_only`:

| `anatomical_only` | image fed to Cellpose |
|---|---|
| 0 | (off -- functional detection) |
| 1 | `max_proj / meanImg` |
| 2 | `meanImg` |
| 3 | **enhanced mean image** |
| 4 | `max_proj` |

suite2p **1.1.0** replaced that integer with a string,
`detection.cellpose_settings.img`, and it accepts **only three** values
(`suite2p/parameters.py`, key `img`):

- `"max_proj / meanImg"`  <- **default**
- `"meanImg"`
- `"max_proj"`

**So: is enhanced mean image still allowed? No** -- not as a detection image. The old
`anatomical_only=3` ("enhanced mean") has no equivalent in `cellpose_settings.img`.
The spatial sharpening it used to do is now a *separate, orthogonal* knob,
`cellpose_settings.highpass_spatial` (flat alias `spatial_hp_cp`), applied on top of
*whichever* of the three images you choose.

The "enhanced mean image" (`meanImgE`) still exists, but only as a **registration
output / GUI view** ("E: mean img (enhanced)") -- it is *not* one of the images passed
to Cellpose.

**This fork's back-compat** (`lbm_suite2p_python/db_settings.py`): the integer
`anatomical_only` still works and is translated to `img`. `anatomical_only=3` warns and
falls back to `1`; legacy `img` strings like `"enhanced_meanImg"` warn and map to the
default. Use `spatial_hp_cp` (`highpass_spatial`) `> 0` to recover the sharpening.

## Exactly how the two source images are built

From `suite2p/detection/detect.py::detection_wrapper` (on the **registered**, time-binned
movie):

```python
meanImg  = mov.mean(axis=0)                                  # mean BEFORE temporal high-pass
mov      = utils.temporal_high_pass_filter(mov, width=highpass_time)
max_proj = mov.max(axis=0)                                   # max AFTER  temporal high-pass
```

Then `suite2p/detection/anatomical.py::select_rois` turns those two into the Cellpose
input `img`:

```python
if img == 'max_proj / meanImg':
    img = np.log(np.maximum(1e-3, max_proj / np.maximum(1e-3, mean_img)))
elif img == 'meanImg':
    img = mean_img
else:                       # 'max_proj'
    img = max_proj.copy()

if highpass_spatial:        # optional sharpening, applied to whichever img above
    img = np.clip(normalize99(img), 0, 1)
    img -= gaussian_filter(img, diameter * highpass_spatial)
    img -= gaussian_filter(img, diameter * highpass_spatial)
```

This notebook reproduces those exact steps on `E:/demo/mk355/raw` so you can see the
literal array handed to Cellpose for every option.

> Note: suite2p computes these from the *registered* movie. We run on the loaded frames
> (no registration), so absolute sharpness differs slightly -- the option logic and
> transforms are identical.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mbo_utilities as mbo

from suite2p.detection.detect import bin_movie
from suite2p.detection.utils import temporal_high_pass_filter
from suite2p.registration import highpass_mean_image  # legacy 'enhanced mean' (meanImgE)
from scipy.ndimage import gaussian_filter

try:
    from cellpose.transforms import normalize99
except Exception:
    def normalize99(img):                  # same 1/99-percentile normalization
        x = img.astype(np.float32)
        lo, hi = np.percentile(x, 1), np.percentile(x, 99)
        return (x - lo) / max(hi - lo, 1e-12)

In [ ]:
# ---- config ----
RAW   = r"E:/demo/mk355/raw"
PLANE = 7        # 0-based z-plane (this dataset has 14)
N_FRAMES = 1000  # frames to load; None = all (full plane is ~1.5 GB in float32)
TAU = 1.0        # indicator timescale (s); sets bin size, as in suite2p
HIGHPASS_TIME = 100      # detection.highpass_time (temporal high-pass width)
HIGHPASS_SPATIAL = 2     # cellpose_settings.highpass_spatial demo value (0 = off)
DIAMETER = 12.0          # cell diameter (px); scales the spatial high-pass sigma

In [ ]:
arr = mbo.imread(RAW)
fs  = float(arr.metadata["fs"])
n   = arr.shape[0] if N_FRAMES is None else min(N_FRAMES, arr.shape[0])
mov = np.asarray(arr[:n, 0, PLANE]).astype(np.float32)   # (T, Ly, Lx)
T, Ly, Lx = mov.shape
print(f"plane {PLANE}: {mov.shape}  fs={fs} Hz")

In [ ]:
# ---- reproduce detection_wrapper's image construction ----
nbins = 5000
bin_size = int(max(1, T // nbins, round(TAU * fs)))       # suite2p's default bin_size
binned = bin_movie(mov, bin_size, yrange=[0, Ly], xrange=[0, Lx], nbins=nbins)

meanImg  = binned.mean(axis=0)                            # mean BEFORE temporal high-pass
filt     = temporal_high_pass_filter(mov=binned.copy(), width=HIGHPASS_TIME)
max_proj = filt.max(axis=0)                               # max AFTER  temporal high-pass
print(f"bin_size={bin_size}  binned={binned.shape}  meanImg={meanImg.shape}")

In [ ]:
def cellpose_input(meanImg, max_proj, img, highpass_spatial=0, diameter=DIAMETER):
    """The exact image select_rois hands to Cellpose for a given `img` option."""
    if img == "max_proj / meanImg":
        out = np.log(np.maximum(1e-3, max_proj / np.maximum(1e-3, meanImg)))
    elif img == "meanImg":
        out = meanImg.copy()
    elif img == "max_proj":
        out = max_proj.copy()
    else:
        raise ValueError(f"suite2p 1.1.0 only accepts 'max_proj / meanImg', "
                         f"'meanImg', 'max_proj' -- got {img!r}")
    if highpass_spatial:
        out = np.clip(normalize99(out), 0, 1)
        out -= gaussian_filter(out, diameter * highpass_spatial)
        out -= gaussian_filter(out, diameter * highpass_spatial)
    return out

In [ ]:
def show(ax, im, title):
    lo, hi = np.percentile(im, 1), np.percentile(im, 99)
    ax.imshow(im, cmap="gray", vmin=lo, vmax=hi)
    ax.set_title(title, fontsize=9)
    ax.axis("off")

fig, axs = plt.subplots(2, 3, figsize=(15, 11))

# row 1: the three images Cellpose can actually be given
show(axs[0, 0], cellpose_input(meanImg, max_proj, "meanImg"),
     "img='meanImg'  (old anatomical_only=2)")
show(axs[0, 1], cellpose_input(meanImg, max_proj, "max_proj"),
     "img='max_proj'  (old anatomical_only=4)")
show(axs[0, 2], cellpose_input(meanImg, max_proj, "max_proj / meanImg"),
     "img='max_proj / meanImg'  [DEFAULT]  (old anatomical_only=1)")

# row 2: the highpass_spatial knob + the legacy 'enhanced mean' for reference
show(axs[1, 0], cellpose_input(meanImg, max_proj, "meanImg", HIGHPASS_SPATIAL),
     f"img='meanImg' + highpass_spatial={HIGHPASS_SPATIAL}")
show(axs[1, 1], cellpose_input(meanImg, max_proj, "max_proj / meanImg", HIGHPASS_SPATIAL),
     f"img='max_proj / meanImg' + highpass_spatial={HIGHPASS_SPATIAL}")
show(axs[1, 2], highpass_mean_image(meanImg.astype("float32"), aspect=1.0),
     "meanImgE  (old anatomical_only=3)\nregistration/GUI only -- NOT a Cellpose option")

fig.suptitle(f"Cellpose detection images -- {RAW}  plane {PLANE}", fontsize=12)
fig.tight_layout()
plt.show()

## Summary

- **Three** Cellpose images in suite2p 1.1.0: `"max_proj / meanImg"` (default), `"meanImg"`,
  `"max_proj"` -- top row above.
- **Enhanced mean image is no longer a detection option.** Old `anatomical_only=3` has no
  `img` equivalent; in this fork it warns and maps to the default `"max_proj / meanImg"`.
- To sharpen any of the three images, set `highpass_spatial`/`spatial_hp_cp` `> 0`
  (bottom-left/middle) -- this is the modern replacement for the old enhancement.
- `meanImgE` (bottom-right) is still computed during registration and shown in the GUI as
  "E: mean img (enhanced)", but it is never passed to Cellpose.